# MLB Underdog Tracker — Dashboard

Running analysis of qualifying underdog bets. Re-run all cells to refresh with latest data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import config
from src.analysis import (
    load_resolved, summary_by_bucket, cumulative_pl,
    best_book_frequency, home_vs_away, break_even_rate
)

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

# Load data
df = load_resolved()
print(f"Resolved games loaded: {len(df)}")

if len(df) == 0:
    print("\nNo resolved games yet. Run the daily pipeline to start collecting data:")
    print("  python -m src.collect_odds")
    print("  python -m src.tracker add")
    print("  (next day)")
    print("  python -m src.collect_scores")
    print("  python -m src.tracker update")

## Performance by Odds Bucket

Key table: win rate vs. break-even rate per bucket. Positive **edge** = actual wins exceed what's needed to break even.

In [ ]:
if len(df) > 0:
    bucket_df = summary_by_bucket(df)
    display(bucket_df.style.format({
        "win_rate": "{:.1%}", "break_even_avg": "{:.1%}",
        "edge": "{:+.1%}", "total_profit": "${:+,.2f}", "roi": "{:+.1f}%"
    }).bar(subset=["edge"], color=["#d65f5f", "#5fba7d"], align="zero"))
else:
    print("Waiting for data...")

## Cumulative P/L Over Time

The big picture — are we trending up, down, or flat?

In [ ]:
if len(df) > 0:
    cpl = cumulative_pl(df)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(cpl["bet_number"], cpl["cum_profit"], linewidth=2, color="#2196F3")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax.fill_between(cpl["bet_number"], cpl["cum_profit"], 0,
                     where=cpl["cum_profit"] >= 0, alpha=0.15, color="green")
    ax.fill_between(cpl["bet_number"], cpl["cum_profit"], 0,
                     where=cpl["cum_profit"] < 0, alpha=0.15, color="red")
    ax.set_xlabel("Bet #")
    ax.set_ylabel("Cumulative P/L ($)")
    ax.set_title("Cumulative Profit/Loss — All Qualifying Underdogs")
    plt.tight_layout()
    plt.show()
else:
    print("Waiting for data...")

## Win Rate vs. Break-Even by Bucket

In [ ]:
if len(df) > 0:
    bucket_df = summary_by_bucket(df)
    buckets_only = bucket_df[bucket_df["bucket"] != "all"]

    if len(buckets_only) > 0:
        fig, ax = plt.subplots(figsize=(8, 5))
        x = range(len(buckets_only))
        width = 0.35
        ax.bar([i - width/2 for i in x], buckets_only["win_rate"],
               width, label="Actual Win Rate", color="#2196F3")
        ax.bar([i + width/2 for i in x], buckets_only["break_even_avg"],
               width, label="Break-Even Rate", color="#FF9800", alpha=0.7)
        ax.set_xticks(list(x))
        ax.set_xticklabels(buckets_only["bucket"])
        ax.set_ylabel("Win Rate")
        ax.set_title("Actual Win Rate vs. Required Break-Even Rate")
        ax.legend()
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
        plt.tight_layout()
        plt.show()
else:
    print("Waiting for data...")

## Home Underdogs vs. Away Underdogs

In [ ]:
if len(df) > 0:
    ha = home_vs_away(df)
    if len(ha) > 0:
        display(ha.style.format({
            "win_rate": "{:.1%}", "total_profit": "${:+,.2f}", "roi": "{:+.1f}%"
        }))
else:
    print("Waiting for data...")

## Best Sportsbook for Underdog Prices

Which book most frequently offers the best underdog moneyline?

In [ ]:
if len(df) > 0:
    bbf = best_book_frequency(df)
    if len(bbf) > 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh(bbf["book"], bbf["times_best"], color="#4CAF50")
        ax.set_xlabel("Times Offering Best Underdog Price")
        ax.set_title("Best Sportsbook Frequency")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("Waiting for data...")

## Recent Bets Log

In [ ]:
if len(df) > 0:
    recent = cumulative_pl(df).tail(20)
    display(recent[["date", "underdog", "dog_odds_best", "bucket",
                     "dog_won", "profit", "cum_profit"]].style.format({
        "profit": "${:+,.2f}", "cum_profit": "${:+,.2f}"
    }).apply(lambda row: ["background-color: #c8e6c9" if row["dog_won"] else
                          "background-color: #ffcdd2"] * len(row), axis=1))
else:
    print("Waiting for data...")